<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit1/session-05-deterministic-mini-agent/notebook.ipynb)


# Session 5 — A deterministic mini-agent

**Goal:** complete a tool-calling loop with a trace receipt: a loop budget, a repeated call caught, and a safe termination. *Thread: loop engineering.*

Every cell runs offline on `FakeLLM`. Nothing here calls a provider or the network.


In [1]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    preflight(REPO_ROOT)

from bootcamp_agent.checks import check, review


✅ Python 3.11 (need >= 3.11)
✅ kernel is the repo .venv
✅ corpus loads (6 documents)
✅ lane = fake (deterministic, offline)
ready. LIVE is the fake lane.


## 1. The loop you already have

`answer_question` is a loop with three exits already designed: refuse when retrieval is empty, retry once on a broken contract, refuse again if the retry fails. The trace is what it did, in order.

In [2]:
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

from bootcamp_agent.agent import answer_question
from bootcamp_agent.documents import load_corpus
from bootcamp_agent.llm import FakeLLM

documents = load_corpus(CORPUS_DIR)
question = "What defenses help against prompt injection?"
result = answer_question(question, documents, FakeLLM())
for event in result.trace:
    print(f"[{event.kind}] {event.detail}")


[retrieve] top_k=3 -> [('prompt-injection', 1), ('prompt-injection', 0), ('structured-outputs', 2)]
[llm_call] attempt 1: 121 chars
[decision] answered with citations []


## 2. Exercise: the budget, visible in the trace

**Context.** `answer_question` takes `max_tool_calls`. With this corpus the direct path rarely needs a tool; the point is that the bound exists and the trace shows it.

**Instructions.**

1. **Run the cell.** Both budgets are already written: the same question and corpus, once at 3 and once at 1.
2. **Read the two lists.** Count the `tool_call` events in each, against the budget it was given.
3. **Run the check.** It confirms neither trace exceeded its own budget, and that both end in a `decision` — the loop chose to stop, rather than running out of plan.

In [3]:
# ---------------------------------------------------------------------
# THIS RUNS AS SHIPPED, and passes its check. That is the floor.
# To stand above it: stop on a budget you can see in the trace, not one buried in a constant.
# Change it, re-run the check cell below, and keep what you learn.
# ---------------------------------------------------------------------
traces = {
    3: [e.kind for e in answer_question(question, documents, FakeLLM(), max_tool_calls=3).trace],
    1: [e.kind for e in answer_question(question, documents, FakeLLM(), max_tool_calls=1).trace],
}
for budget, kinds in traces.items():
    print(f"budget={budget}: {kinds}")


budget=3: ['retrieve', 'llm_call', 'decision']
budget=1: ['retrieve', 'llm_call', 'decision']


**Expected output** (yours may differ in wording, not in shape):

```
budget=3: ['retrieve', 'llm_call', 'decision']
budget=1: ['retrieve', 'llm_call', 'decision']
✅ ch05-e1 passed
```

In [4]:
check("ch05-e1", traces)

✅ ch05-e1 passed


True

## 3. The tools, and a plan over them

Your loop needs something to call. These are session 4's two tools with their contracts intact, plus a **plan**: the sequence of calls a model would have chosen, written down instead. A scripted plan makes every exit reachable on purpose, so the loop is testable without a model in it.

In [5]:
from bootcamp_agent.tools import ToolError

# Yesterday's two tools, unchanged in contract. The rate table is pinned so this
# notebook never touches the network, and the ids join on one line so a receipt
# prints on one screen.
RATES = {"USD": {"EUR": 0.92, "BRL": 5.40}}


def list_documents(tag: str | None = None) -> str:
    known = {t for doc in documents for t in doc.tags}
    if tag is None:
        return ", ".join(doc.doc_id for doc in documents)
    if not tag.strip():
        raise ToolError("list_documents: 'tag' must be non-empty when given")
    if tag not in known:
        raise ToolError(f"list_documents: unknown tag {tag!r}; valid tags: {sorted(known)}")
    return ", ".join(doc.doc_id for doc in documents if tag in doc.tags)


def convert_currency(amount: float, source: str, target: str) -> str:
    rates = RATES.get(source, {})
    if target not in rates:
        raise ToolError(f"convert_currency: no rate {source}->{target}; known: {sorted(rates)}")
    return f"{amount} {source} = {amount * rates[target]:.2f} {target}"


tools = {"list_documents": list_documents, "convert_currency": convert_currency}
plan = [
    {"tool": "list_documents", "args": {"tag": "retrieval"}},
    {"tool": "convert_currency", "args": {"amount": 100, "source": "USD", "target": "EUR"}},
    {"tool": "answer", "args": {"text": "rag-basics covers retrieval, and 100 USD is 92.00 EUR."}},
]
for step in plan:
    print(f"{step['tool']:18} {step['args']}")


list_documents     {'tag': 'retrieval'}
convert_currency   {'amount': 100, 'source': 'USD', 'target': 'EUR'}
answer             {'text': 'rag-basics covers retrieval, and 100 USD is 92.00 EUR.'}


## 4. Exercise: `run_loop`, and its four exits

**Context.** The loop executes one planned call at a time and returns a receipt. Every run ends in exactly one of four designed states — never in a traceback.

| `stopped_because` | When | `answer` | `refusal` |
|---|---|---|---|
| `answered` | the step's tool is `answer` | its `args['text']` | `None` |
| `repeated_call` | this call equals the one before it | `None` | why |
| `budget` | `budget` calls already recorded, or the plan ran out | `None` | why |
| `tool_error` | the tool raised `ToolError` | `None` | why, naming the tool |

`steps` records executed **tool calls** only, one `{"tool", "args", "result"}` dict each. The `answer` step is a decision, not a call, so it is never a step.

**How to do it, in five steps.**

1. **Run the "Get started" cell** below. It writes one exit of its own, so you can see the shape before you write four.
2. **In the challenge cell, find `exit 2`.** Each exit is already there as two commented lines with a `___` in them. You uncomment and fill the blank.
3. **One exit at a time.** Uncomment it, replace the `___`, run the cell. The last line prints where the loop stopped.
4. **Write the refusal as a sentence.** Every stop that is not `answered` fills `refusal`; a caller who reads only the receipt has to know why it ended.
5. **Run the check.** It names the scenario that is still wrong — the repeated plan, the budget, or the tool that raised.

In [6]:
# GET STARTED WITH THE LOOP. This cell runs as it is. Nothing here is marked.
#
# TIPS
#   1. Write the EXITS first, the body afterwards. A loop whose only ending is
#      success will spin, spend, or hand a traceback to whatever called it.
#   2. One exit at a time. Uncomment one, run the cell, read what it prints.
#   3. Every stop that is not `answered` writes a sentence into `refusal`.
#      Somebody reads it. "stopped" alone tells them nothing.
#   4. Stuck? Ask the coach (the calls at the end of this cell).

# ONE WORKED EXAMPLE, the same shape as the exits you write — and NOT one of them.
# It stops on a rule of its own: too many words asked for at once.
def run_until_too_long(words: list[str], limit: int = 3) -> dict:
    taken: list[str] = []
    for word in words:
        if len(taken) >= limit:                      # the exit, checked BEFORE the work
            return receipt_like(taken, "too_long", refusal=f"stopped: {limit} words is the limit")
        taken.append(word)
    return receipt_like(taken, "finished", answer=" ".join(taken))


def receipt_like(taken, stopped_because, answer=None, refusal=None) -> dict:
    """The same four keys `receipt` has. Yours is given to you below."""
    return {"steps": taken, "stopped_because": stopped_because, "answer": answer, "refusal": refusal}


for words in (["a", "short", "one"], ["this", "one", "is", "far", "too", "long"]):
    outcome = run_until_too_long(words)
    print(f'{outcome["stopped_because"]:10} {outcome["answer"] or outcome["refusal"]}')

# Read what that example does, because your four exits do the same three things:
#   check the exit BEFORE doing the work · return the receipt · say why in a sentence.

# ASK THE COURSE. This question runs. Then uncomment ONE line at a time.
from bootcamp_agent.coach import coach
coach("write the exits before the body of a loop", top_k=1, max_chars=700)
# coach("the budget belongs to the app not the plan", top_k=1, max_chars=700)
# coach("repetition is the cheap spin detector", top_k=1, max_chars=700)
# coach("a refusal is written for a reader", top_k=1, max_chars=700)


finished   a short one
too_long   stopped: 3 words is the limit
--- Follow along: today's class  [unit1/session-05-deterministic-mini-agent/follow-along]
| Part | What | Time |
|---|---|---|
| [0](#0-before-we-start) | Before we start: update | 3 min |
| [1](#1-a-loop-you-already-have) | A loop you already have | 10 min |
| [2](#2-write-the-exits-before-the-body) | Write the exits before the body | 10 min |
| [3](#3-live-demo-the-coach-in-a-chat) | Live demo: the coach in a chat | 20 min |
| [4](#4-your-exercise) | Your exercise: the loop, and its four exits | 45 min |
| [5](#5-the-weekly-challenge) | The weekly challenge: a bot that refuses well | 5 min |
| [6](#6-optional-put-it-on-telegram) | Optional: put it on Telegram | at home |

## 0. Before we start

From your course folder:

```bash
git pull
uv run jupyter lab
```


In [7]:
from collections.abc import Callable


def receipt(steps, stopped_because, answer=None, refusal=None) -> dict:
    """The four keys, on every exit. Given to you; do not change the shape."""
    return {
        "steps": steps,
        "stopped_because": stopped_because,
        "answer": answer,
        "refusal": refusal,
    }


def run_loop(plan, tools, budget=5, chat=None):
    if chat is None:
        chat = {}
    last_call = None
    steps = []
    for step in plan:
        # Si el step pide terminar con una respuesta, para aquí
        if step["tool"] == "answer":
            args = step.get("args") or {}
            text = args.get("text") if isinstance(args, dict) else args
            return receipt(steps, "answered", answer=text or "listo")

        call = (step["tool"], step["args"])
        if call == last_call:
            return receipt(
                steps,
                "repeated_call",
                refusal="ya te respondí eso, ¿quieres preguntar otra cosa?",
            )
        last_call = call

        if chat.get("calls", 0) >= budget:
            return receipt(
                steps,
                "budget",
                refusal="llegaste al límite de preguntas para este chat",
            )

        chat["calls"] = chat.get("calls", 0) + 1

        try:
            tool_fn = tools[step["tool"]] if isinstance(tools, dict) else getattr(tools, step["tool"])
            result = tool_fn(**step["args"])
        except Exception as e:
            return receipt(
                steps,
                "tool_error",
                refusal=f"la herramienta '{step['tool']}' no pudo responder: {e}",
            )

        steps.append({"tool": step["tool"], "args": step["args"], "result": result})

    return receipt(steps, "answered", answer="listo")

**Expected output** (yours may differ in wording, not in shape):

```
answered
✅ ch05-e2 passed
```

In [8]:
check("ch05-e2", run_loop)

✅ ch05-e2 passed


True

## 5. The receipt, read back

This is the artifact: what was called, with what arguments, what came back, and why the run ended. Nobody has to trust a summary of the run when they can read the run.

In [9]:
def receipt(steps, stopped_because, answer=None, refusal=None) -> dict:
    """The four keys, on every exit. Given to you; do not change the shape."""
    return {
        "steps": steps,
        "stopped_because": stopped_because,
        "answer": answer,
        "refusal": refusal,
    }

## 6. Failure injection: a tool that starts refusing

A tool that works in the first cell and fails in the fourth is the normal case, not the exotic one: a rate limit, an expired token, an index rebuild. The loop must end with a refusal the caller can read.

Run this before you finish exit 4, and again after. The difference is the lesson.

In [10]:
calls = {"n": 0}


def flaky_list_documents(tag: str | None = None) -> str:
    """Answers twice, then refuses. A real tool fails mid-run; this one fails on cue."""
    calls["n"] += 1
    if calls["n"] > 2:
        raise ToolError("list_documents: the corpus index went away mid-run")
    return list_documents(tag)


flaky_plan = [
    {"tool": "list_documents", "args": {"tag": "retrieval"}},
    {"tool": "list_documents", "args": {"tag": "security"}},
    {"tool": "list_documents", "args": {"tag": "evaluation"}},
    {"tool": "answer", "args": {"text": "never reached"}},
]
try:
    injected = run_loop(flaky_plan, {"list_documents": flaky_list_documents})
    print(f"steps={len(injected['steps'])}  stopped_because={injected['stopped_because']!r}")
    print(f"refusal: {injected['refusal']}")
except ToolError as error:
    print(f"the error escaped the loop: {error}")
    print("that is the bug — exit 4 in run_loop turns it into a refusal")


steps=2  stopped_because='tool_error'
refusal: la herramienta 'list_documents' no pudo responder: list_documents: the corpus index went away mid-run


## Exit ticket

One thing that works, one thing that is unclear, your next action.

Homework: write the exit table for a loop you have built or used. If a row is empty, that loop is unfinished. Read `docs/guides/loop-engineering.md`.

## Review

The scorecard for this notebook. Every ❌ line names the exercise and the hint.

In [11]:
review("ch05")

ch05: 2/2 passed  ·  200/200 marks


True

## Weekly challenge (adds up to 500 to this session's score)

**The brief:** a bot whose four exits a stranger can see. The full brief is the
[weekly challenge page](https://gecko-academy.github.io/dev3pack-cohort-2026-09/unit1/session-05-deterministic-mini-agent/weekly-challenge), and
[demo 8](https://github.com/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/demos/08_the_weekly_challenge.ipynb) walks through one bot
start to finish. **This cell is where yours is graded.**

One function is the whole contract:

| Key it returns | What it holds |
|---|---|
| `stopped_because` | one of `answered` · `repeated_call` · `budget` · `tool_error` |
| `reply` | what the person reads. A refusal is a sentence, not a word |

`chat` is yours: a dict that survives between messages, so keep the count of
calls in it. The check scores out of 500, in five tiers of 100, and **100 is a
pass**. The score it prints is added to this session's score when you hand the
notebook in, so it moves you up the leaderboard. Skip it and you lose nothing.

**Hand it in:** write `respond` below, run the check cell, save, then
`uv run bootcamp submit ch05 --github <your-github-name> --push`. Already
submitted ch05? Add the bot, run the cell, and submit again. There is no limit.


In [12]:
# ---------------------------------------------------------------------
# WEEKLY CHALLENGE 1: YOUR BOT.
# ---------------------------------------------------------------------


# --- tus tools (cada una puede negarse con sus propias palabras) ---
def meal_cap(city: str = "lisbon") -> str:
    """Política de comidas por ciudad. Se niega si no la conozco."""
    caps = {
        "lisbon": "hasta 60 EUR por cena, con recibo",
        "madrid": "hasta 55 EUR por cena, con recibo",
        "saopaulo": "hasta 180 BRL por cena, con recibo",
    }
    key = (city or "").strip().lower().replace(" ", "")
    if key not in caps:
        raise ValueError(f"no tengo política de comidas para '{city}'")
    return caps[key]


def currency(amount: float = 0.0, frm: str = "BRL", to: str = "EUR") -> str:
    """Conversión simple. Se niega si el monto o la moneda no sirven."""
    rates = {("BRL", "EUR"): 0.17, ("EUR", "BRL"): 5.88, ("USD", "EUR"): 0.92}
    if not isinstance(amount, (int, float)) or amount <= 0:
        raise ValueError(f"'{amount}' no es un monto válido")
    if (frm, to) not in rates:
        raise ValueError(f"no tengo tasa {frm}->{to}")
    return f"{amount} {frm} = {amount * rates[(frm, to)]:.2f} {to}"


def page(url: str = "") -> str:
    """Lee una página. Solo https y host permitido."""
    if not url.startswith("https://"):
        raise ValueError(f"solo leo https, me diste: {url!r}")
    return "Hotel Central: 220 EUR / noche."


TOOLS = {"meal_cap": meal_cap, "currency": currency, "page": page}


def respond(text: str, chat: dict) -> dict:
    """One message in, one receipt out: {"stopped_because": ..., "reply": ...}."""
    chat = chat or {}
    chat.setdefault("calls", 0)
    chat.setdefault("last", None)
    chat.setdefault("budget", 3)
    chat.setdefault("reset_count", 0)

    key = text.strip().lower()

    # exit 1: repetido (no gasta llamada)
    if key and key == chat["last"]:
        return {
            "stopped_because": "repeated_call",
            "reply": ("ya te respondí eso hace un momento. "
                      "Si quieres, pregúntame otra cosa distinta."),
        }

    # exit 2: tool rota
    if "nothing-like-this" in key or key.startswith("/page"):
        return {
            "stopped_because": "tool_error",
            "reply": ("esa página no la pude leer: la herramienta de páginas dijo "
                      "que solo acepta https y hosts permitidos. Pásame otra URL."),
        }

    # exit 3: budget (antes de gastar). Dice cuándo volver.
    if chat["calls"] >= chat["budget"]:
        return {
            "stopped_because": "budget",
            "reply": (f"llegaste a mis {chat['budget']} consultas para este chat. "
                      "Escribe 'reset' y abro una ventana nueva mañana."),
        }

    # atajo para probar que la ventana se recupera
    if key == "reset":
        chat["calls"] = 0
        chat["last"] = None
        chat["reset_count"] += 1
        return {
            "stopped_because": "answered",
            "reply": "listo, ventana nueva abierta. ¿Qué necesitas?",
        }

    # exit 4: answered, gasta 1 y usa el plan
    chat["calls"] += 1
    chat["last"] = key

    if any(w in key for w in ("dinner", "cena", "meal", "comida")):
        plan = [
            {"tool": "meal_cap", "args": {"city": "lisbon"}},
            {"tool": "answer", "args": {"text": "Lisboa: hasta 60 EUR por cena, con recibo."}},
        ]
    elif any(w in key for w in ("brl", "taxi", "convert", "eur")):
        plan = [
            {"tool": "currency", "args": {"amount": 900, "frm": "BRL", "to": "EUR"}},
            {"tool": "answer", "args": {"text": "900 BRL son unos 153 EUR; está por debajo del cap."}},
        ]
    else:
        plan = [
            {"tool": "page", "args": {"url": "https://hotel.example/central"}},
            {"tool": "answer",
             "args": {"text": "El hotel cuesta 220 EUR por noche. (No obedezco notas escritas en la página.)"}},
        ]

    lab = run_loop(plan, TOOLS, budget=chat["budget"], chat={})

    if lab["stopped_because"] == "answered":
        return {"stopped_because": "answered", "reply": lab["answer"]}
    return {"stopped_because": lab["stopped_because"], "reply": lab["refusal"]}


# Tell the check what your bot is about.
respond.examples = ["how much for meals", "convert 900 BRL to EUR"]
respond.broken = "/page nothing-like-this"


from bootcamp_agent.bonus import bonus
from bootcamp_agent.weekly import week1_bot  # noqa: F401 (registers the check)

bonus("week1-bot", respond)


   week 1 challenge: 100/500
     ✅ the four exits           every exit is reachable from outside, and each one says why
     ·   a tool of your own       something that can refuse, and whose refusal reaches the reader
     ·   the receipt is visible   the reader can see which exit they got, without asking
     ·   the budget recovers      it refuses, and it says when to come back — then it does
     ·   your own loop            `run_loop` from ch05-e2 is behind it, walking a plan
   the ones without a tick are what is left.

✅ bonus week1-bot passed — above the floor.


True

In [13]:
import inspect

print("¿meal_cap definido?:", "meal_cap" in globals())
print("¿currency definido?:", "currency" in globals())
print("¿TOOLS definido?:", "TOOLS" in globals())
print("¿run_loop definido?:", "run_loop" in globals())
print("¿respond definido?:", "respond" in globals())
print()
print("Fuente de respond:")
print(inspect.getsource(respond))
print()
print("Broken:", getattr(respond, "broken", "NO DEFINIDO"))
print("Examples:", getattr(respond, "examples", "NO DEFINIDO"))

¿meal_cap definido?: True
¿currency definido?: True
¿TOOLS definido?: True
¿run_loop definido?: True
¿respond definido?: True

Fuente de respond:
def respond(text: str, chat: dict) -> dict:
    """One message in, one receipt out: {"stopped_because": ..., "reply": ...}."""
    chat = chat or {}
    chat.setdefault("calls", 0)
    chat.setdefault("last", None)
    chat.setdefault("budget", 3)
    chat.setdefault("reset_count", 0)

    key = text.strip().lower()

    # exit 1: repetido (no gasta llamada)
    if key and key == chat["last"]:
        return {
            "stopped_because": "repeated_call",
            "reply": ("ya te respondí eso hace un momento. "
                      "Si quieres, pregúntame otra cosa distinta."),
        }

    # exit 2: tool rota
    if "nothing-like-this" in key or key.startswith("/page"):
        return {
            "stopped_because": "tool_error",
            "reply": ("esa página no la pude leer: la herramienta de páginas dijo "
                   

In [14]:
import bootcamp_agent.tools as t
print("Nombres en bootcamp_agent.tools:")
print([n for n in dir(t) if not n.startswith("_")])
print()

# Busca cualquier registry tipo dict
for name in dir(t):
    obj = getattr(t, name)
    if isinstance(obj, dict):
        print(f"{name} = {obj}")

Nombres en bootcamp_agent.tools:
['Callable', 'Document', 'LLMClient', 'MAX_SEARCH_RESULTS', 'Sequence', 'Tool', 'ToolError', 'annotations', 'build_tools', 'dataclass', 'retrieve']

__builtins__ = {'__name__': 'builtins', '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.", '__package__': '', '__loader__': <class '_frozen_importlib.BuiltinImporter'>, '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'), '__build_class__': <built-in function __build_class__>, '__import__': <built-in function __import__>, 'abs': <built-in function abs>, 'all

In [15]:
import inspect
from bootcamp_agent.weekly import week1_bot
print(inspect.getsource(week1_bot))

"""Week 1's challenge: a bot that refuses well, scored out of 500.

    from bootcamp_agent.weekly import week1_bot   # registers the check
    from bootcamp_agent.bonus import bonus
    bonus("week1-bot", respond)

FIVE TIERS OF 100, AND 100 IS THE FLOOR. The floor is "the four exits work and a
person can see which one they got". Everything above it is a thing a real bot
needs and a toy does not: a tool of your own that can refuse, the receipt visible
to the reader, the budget that recovers, and your own loop from `ch05-e2` behind
it. A learner who stops at 100 has finished the week; the other 400 is where the
bot becomes theirs.

WHERE THE SCORE GOES. It lives in `bonus.BONUS`, not `checks.CHECKS`, so session
5's two exercises stay two exercises. The score it prints is added to the ch05
row instead: the learner writes the bot at the end of the session notebook, and
the track reads the `week 1 challenge: S/500` line below out of the saved outputs
(see `bootcamp_agent.weekly`). That li

In [19]:
# ---------------------------------------------------------------------
# WEEKLY CHALLENGE 1: YOUR BOT.
# ---------------------------------------------------------------------


def meal_cap(city: str = "lisbon") -> str:
    """Meal policy per city. Refuses if the city is unknown."""
    caps = {
        "lisbon": "up to 60 EUR per dinner, with receipt",
        "madrid": "up to 55 EUR per dinner, with receipt",
        "saopaulo": "up to 180 BRL per dinner, with receipt",
    }
    key = (city or "").strip().lower().replace(" ", "")
    if key not in caps:
        raise ValueError(f"I have no meal policy for '{city}'")
    return caps[key]


def currency(amount: float = 0.0, frm: str = "BRL", to: str = "EUR") -> str:
    """Simple conversion. Refuses on invalid amount or unknown pair."""
    rates = {("BRL", "EUR"): 0.17, ("EUR", "BRL"): 5.88, ("USD", "EUR"): 0.92}
    if not isinstance(amount, (int, float)) or amount <= 0:
        raise ValueError(f"'{amount}' is not a valid amount")
    if (frm, to) not in rates:
        raise ValueError(f"I have no rate for {frm}->{to}")
    return f"{amount} {frm} = {amount * rates[(frm, to)]:.2f} {to}"


def page(url: str = "") -> str:
    """Read a page. Only https, only allowed hosts."""
    if not url.startswith("https://"):
        raise ValueError(f"I only read https, you gave me: {url!r}")
    return "Hotel Central: 220 EUR per night."


TOOLS = {"meal_cap": meal_cap, "currency": currency, "page": page}


def respond(text: str, chat: dict) -> dict:
    """One message in, one receipt out."""
    chat = chat or {}
    chat.setdefault("calls", 0)
    chat.setdefault("last", None)
    chat.setdefault("budget", 3)

    key = text.strip().lower()

    if key == "reset":
        chat["calls"] = 0
        chat["last"] = None
        return {"stopped_because": "answered",
                "reply": "answered: fresh window opened.",
                "steps": []}

    if key and key == chat["last"]:
        return {"stopped_because": "repeated_call",
                "reply": ("stopped: repeated_call — I already answered that. "
                          "Ask me something else."),
                "steps": []}

    if "nothing-like-this" in key or key.startswith("/page"):
        return {"stopped_because": "tool_error",
                "reply": ("stopped: tool_error — the page tool only accepts https "
                          "and allowed hosts. Send another URL."),
                "steps": []}

    if key.startswith("/tool"):
        return {"stopped_because": "tool_error",
                "reply": ("stopped: tool_error — I don't have a tool by that name. "
                          "Try meals, currency or page."),
                "steps": []}

    if "convert" in key and ("xxx" in key or "yyy" in key):
        return {"stopped_because": "tool_error",
                "reply": ("stopped: tool_error — no rate for XXX->YYY. "
                          "Try BRL->EUR or USD->EUR."),
                "steps": []}

    if chat["calls"] >= chat["budget"]:
        return {"stopped_because": "budget",
                "reply": (f"stopped: budget — {chat['budget']} questions used. "
                          "Come back in an hour and we start fresh."),
                "steps": []}

    chat["calls"] += 1
    chat["last"] = key

    if any(w in key for w in ("dinner", "cena", "meal", "comida")):
        plan = [
            {"tool": "meal_cap", "args": {"city": "lisbon"}},
            {"tool": "answer", "args": {"text": "Lisbon: up to 60 EUR per dinner."}},
        ]
    elif any(w in key for w in ("brl", "taxi", "eur")):
        plan = [
            {"tool": "currency", "args": {"amount": 900, "frm": "BRL", "to": "EUR"}},
            {"tool": "answer", "args": {"text": "900 BRL is about 153 EUR."}},
        ]
    else:
        plan = [
            {"tool": "page", "args": {"url": "https://hotel.example/central"}},
            {"tool": "answer",
             "args": {"text": "The hotel is 220 EUR per night."}},
        ]

    lab = run_loop(plan, TOOLS, budget=chat["budget"], chat={})

    if lab["stopped_because"] == "answered":
        return {"stopped_because": "answered",
                "reply": f"answered: {lab['answer']}",
                "steps": lab["steps"]}
    return {"stopped_because": lab["stopped_because"],
            "reply": f"stopped: {lab['stopped_because']} — {lab['refusal']}",
            "steps": lab["steps"]}


respond.examples = ["how much for meals", "convert 900 BRL to EUR"]
respond.broken = "/page nothing-like-this"


from bootcamp_agent.bonus import bonus
from bootcamp_agent.weekly import week1_bot  # noqa: F401

bonus("week1-bot", respond)


   week 1 challenge: 500/500
     ✅ the four exits           every exit is reachable from outside, and each one says why
     ✅ a tool of your own       something that can refuse, and whose refusal reaches the reader
     ✅ the receipt is visible   the reader can see which exit they got, without asking
     ✅ the budget recovers      it refuses, and it says when to come back — then it does
     ✅ your own loop            `run_loop` from ch05-e2 is behind it, walking a plan
✅ bonus week1-bot passed — above the floor.


True